In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("practice_2Apr2025_144700").getOrCreate()

25/04/02 14:48:10 WARN Utils: Your hostname, TTNPL-8203 resolves to a loopback address: 127.0.1.1; using 10.1.209.177 instead (on interface wlp0s20f3)
25/04/02 14:48:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/02 14:48:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [62]:
from pyspark.sql.functions import col,when,count,year,month,desc,asc,sum
from pyspark.sql.types import *

In [39]:
dummy_df = spark.read.csv("file:///home/hdoop/notebooks/data/spark_practice2/2apr2025/myTranformationPractice/dummy_data.csv",header = True,inferSchema=True)

In [67]:
dummy_df.show(n = 100,truncate = False)

+-----------+----------------+---+------+---------+--------------+---------------+------+--------+-------------+
|Customer_ID|Name            |Age|Gender|Country  |Product       |Category       |Price |Quantity|Purchase_Date|
+-----------+----------------+---+------+---------+--------------+---------------+------+--------+-------------+
|101        |John Doe        |28 |Male  |USA      |Laptop        |Electronics    |1200.5|1       |2024-01-15   |
|102        |Alice Smith     |35 |Female|Canada   |Smartphone    |Electronics    |800.75|2       |2024-02-20   |
|103        |Bob Johnson     |45 |Male  |UK       |Headphones    |Accessories    |150.0 |3       |2024-03-05   |
|104        |Emma Davis      |52 |Female|Germany  |Tablet        |Electronics    |400.25|1       |2024-04-10   |
|105        |Michael Brown   |30 |Male  |India    |Smartwatch    |Accessories    |250.99|2       |2024-05-18   |
|106        |Sophia Wilson   |40 |Female|USA      |Monitor       |Electronics    |300.49|1      

In [41]:
dummy_df.printSchema()

root
 |-- Customer_ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Purchase_Date: date (nullable = true)



In [43]:
from pyspark.sql.functions import col, sum, when

dummy_df.select([sum(when(col(c).isNull() | (col(c) == ""), 1).otherwise(0)).alias(c) for c in dummy_df.columns]).show()


+-----------+----+---+------+-------+-------+--------+-----+--------+-------------+
|Customer_ID|Name|Age|Gender|Country|Product|Category|Price|Quantity|Purchase_Date|
+-----------+----+---+------+-------+-------+--------+-----+--------+-------------+
|          0|   0|  0|     0|      0|      0|       0|    0|       0|            0|
+-----------+----+---+------+-------+-------+--------+-----+--------+-------------+



#### add new col
1.Total_cost
2.Month
3.Year

In [45]:
df1 = dummy_df.withColumn("Total_cost",col("Price")*col("Quantity"))

In [50]:
df2 = df1.withColumn("Year",year(col("Purchase_Date"))).withColumn("Month",month(col("Purchase_Date")))


In [57]:
df3 = df2.withColumn("Category",when((col("Age")>=18)&(col("Age")<=30),"Young")\
                     .when(col("Age")<=50,"Middle-Aged")\
                     .when(col("Age")>50,"Senior")
                    )

In [66]:
df3.groupBy("Product").agg(
    sum("Quantity").alias("Total_Quantity"),
    count("*")
).orderBy(col("Total_Quantity").desc()).show(n=5)

+----------+--------------+--------+
|   Product|Total_Quantity|count(1)|
+----------+--------------+--------+
|  Keyboard|             4|       1|
|Headphones|             3|       1|
|Smartwatch|             2|       1|
|     Mouse|             2|       1|
|Smartphone|             2|       1|
+----------+--------------+--------+
only showing top 5 rows

